# Purchasing power across Portuguese municipalities

**Real data, 278 units.** Columbus has 49 neighbourhoods, which is enough to
demonstrate an estimator and not enough for the asymptotics any of them assume.
This notebook runs the same sequence on the 278 contiguous municipalities of
mainland Portugal, where the sample size is no longer the thing you have to
apologise for.

The question is plain: **why is purchasing power higher in some municipalities
than others, and does space do any of the explaining?** Per capita purchasing
power is an index with Portugal = 100. Lisboa is at 181 and Tabuaço at 64, and
the gap is not scattered at random across the map.

Four things happen here that cannot happen on 49 units:

1. The **regime split rejects.** In notebook `05` the Chow test on Columbus
   says a north/south split is not real. Here a coastal/interior split *is*,
   and the mechanism is specific: on the coast purchasing power tracks density
   and ignores ageing; inland it is the other way round.
2. **The specification tests agree with the likelihood.** The robust LM pair
   points at the error model, and SEM then beats SAR on AIC by 34 points with
   a Vuong test to match. On Columbus the battery lands ambiguously.
3. **Queen against rook stops being interesting.** The two weight matrices
   differ by two links out of 739, and ρ̂ moves by 0.009.
4. **Conley standard errors have real coordinates** to work with, and a
   bandwidth sweep shows which conclusions survive and which do not.

Data and licensing are in `data/portugal_municipalities_PROVENANCE.md`. Both
INE's and DGT's credits are conditions of their licences; keep them with
anything you publish from this.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using Printf, Statistics, LinearAlgebra, Random
using StatsAPI: vcov

const DATA = joinpath(@__DIR__, "data")

## 1. Two files, one key, and a join you should check rather than trust

The attributes and the weights arrive separately: a CSV of 308 municipalities
and a GeoDa GAL file over the 278 on the mainland. They are joined on the
**DICO**, the official district-and-municipality code, and both are written in
DICO order so the join is a filter and a sort rather than a lookup.

That is exactly the kind of claim worth verifying in code, because when it is
wrong nothing throws — rows simply line up against the wrong neighbours and
every number afterwards is quietly meaningless. `read_gal` returns the ids it
read, so the check costs one line.

In [ ]:
lines = readlines(joinpath(DATA, "portugal_municipalities.csv"))
hdr   = split(strip(lines[1]), ',')
rows  = [split(strip(l), ',') for l in lines[2:end] if !isempty(strip(l))]
col(name) = [r[findfirst(==(name), hdr)] for r in rows]
num(name) = [parse(Float64, s) for s in col(name)]

mainland = col("mainland") .== "true"
keep     = findall(mainland)

dico  = col("dico")[keep]
place = col("name")[keep]
distr = col("district_name")[keep]

y = num("purchasing_power")[keep]
X = hcat(log.(num("density")[keep]), num("ageing_ratio")[keep])
vn = ["log_density", "ageing_ratio"]
N  = length(keep)

W,  wid = read_gal(joinpath(DATA, "portugal_municipalities_queen.gal"))
Wr, rid = read_gal(joinpath(DATA, "portugal_municipalities_rook.gal"))

@printf("CSV rows %d, mainland %d, islands %d\n",
        length(rows), N, length(rows) - N)
@printf("GAL is %d x %d; ids match the CSV in order: queen %s, rook %s\n",
        size(W, 1), size(W, 2), wid == dico, rid == dico)
@printf("rows of W sum to one: %s\n",
        all(x -> isapprox(x, 1.0), vec(sum(W; dims = 2))))

The 30 island municipalities are dropped here, and the `mainland` column exists
so that the choice is visible. Madeira and the Azores have figures and **no
land neighbours**: a contiguity matrix over all 308 has 30 entirely zero rows,
and row-standardising divides by zero. Dropping them is one defensible answer;
a k-nearest or distance weighting that gives every unit neighbours is another.
Silently inheriting a singular matrix is not.

Note what `read_gal` did: the file on disk holds **binary** contiguity, and the
row-standardising happened on the way in. That ordering matters if you ever
subset further — standardise last, because dropping a unit changes its
neighbours' degrees.

In [ ]:
@printf("purchasing power: mean %.2f, median %.2f\n", mean(y), median(y))
ord = sortperm(y; rev = true)
println("\nhighest:")
for i in ord[1:6]
    @printf("  %-22s %-18s %7.2f\n", place[i], distr[i], y[i])
end
println("lowest:")
for i in ord[end-3:end]
    @printf("  %-22s %-18s %7.2f\n", place[i], distr[i], y[i])
end
@printf("\ndensity  median %7.1f  max %7.1f (%s)\n",
        median(exp.(X[:, 1])), maximum(exp.(X[:, 1])),
        place[argmax(X[:, 1])])
@printf("ageing   median %7.1f  max %7.1f (%s)  min %7.1f (%s)\n",
        median(X[:, 2]), maximum(X[:, 2]), place[argmax(X[:, 2])],
        minimum(X[:, 2]), place[argmin(X[:, 2])])

Lisboa at 181.4 and Porto at 162.2 against a municipal mean of 82.9 — note that
the mean is *below* the national index of 100, which is what you would expect
when the population is concentrated in the few municipalities at the top. The
bottom of the table is Tabuaço, Penamacor, Vimioso and Vinhais, all interior,
all around 64.

The spread in the regressors is wider than the spread in the outcome. Density
runs from single digits to 8,266 per km² in Amadora, which is why it enters in
logs. The ageing ratio — people aged 65+ per 100 aged 0–14 — runs from 106 in
Montijo to 671 in Alcoutim, so some municipalities have nearly seven pensioners
per child.

## 2. Is there spatial dependence at all?

Before any model, ask whether the map carries information. Moran's I on each
variable is the cheapest possible answer, and here it is not a marginal call.

In [ ]:
for (nm, v) in (("purchasing power", y), ("log density", X[:, 1]),
                ("ageing ratio", X[:, 2]))
    println(rpad(nm, 18), moran_global(v, W))
end

All three are strongly clustered: I = 0.445 for purchasing power (z = 12.1),
0.766 for log density (z = 20.4) and 0.578 for the ageing ratio (z = 15.4)
against an expectation of −0.0036. Portugal's economic geography is not subtle.

This tells you dependence exists. It does **not** tell you the outcome needs a
spatial model: if purchasing power is clustered only because density is
clustered, ordinary regression on density would absorb it. The next step is to
check the residuals, not the raw variable.

## 3. OLS, and the diagnostic that refuses to let you stop there

In [ ]:
Xc    = hcat(ones(N), X)
b_ols = Xc \ y
e_ols = y - Xc * b_ols
r2    = 1 - sum(abs2, e_ols) / sum(abs2, y .- mean(y))

@printf("OLS   intercept %8.3f   log_density %7.3f   ageing %8.4f   R2 %.4f\n",
        b_ols[1], b_ols[2], b_ols[3], r2)
println("\nMoran's I on the OLS residuals:")
println(moran_global(e_ols, W))

Both signs are what the story predicts — denser municipalities richer, older
ones poorer — and R² = 0.44 on two regressors is respectable. If we stopped
here we would publish it.

The residuals say otherwise. Moran's I on them is **0.336 with z = 9.1**: the
clustering in purchasing power is not merely inherited from clustering in
density and age. Something spatial is left over, and OLS standard errors are
computed as though it were not there.

In [ ]:
lms = lm_tests(y, Xc, W)
for k in (:lag, :error, :robust_lag, :robust_error)
    println(getfield(lms, k))
end

Read this in order, because the order is the whole point.

The two **classical** statistics both reject hard — LM-lag 40.0, LM-error 77.6.
That is the usual situation and it is uninformative on its own: each test is
computed assuming the *other* kind of dependence is absent, so when either kind
is present both tend to fire.

The **robust** pair is the one that discriminates. Robust LM-error is 46.4
(p ≈ 1e-11); robust LM-lag is 8.8 (p = 0.003). Both still reject, so this is
not a clean verdict — but the error statistic is five times the lag statistic,
and the battery is pointing at a **spatial error** process rather than a
spatial lag. Hold that prediction: the likelihood will be asked to agree with
it in the next section, and it is worth knowing in advance what would count as
agreement.

## 4. Estimation, and a specification test that was right

SAR puts the dependence in the outcome: rich municipalities make their
neighbours rich. SEM puts it in the disturbance: whatever is omitted is itself
spatially smooth. They are different claims about the world and they are not
nested in each other.

In [ ]:
m_sar = fit_spatial(SAR(), W, X, y; varnames = vn, method = ExactLU())
m_sem = fit_spatial_error(W, X, y;  varnames = vn, method = ExactLU())
m_sdm = fit_spatial(SDM(), W, X, y; varnames = vn, method = ExactLU())

println(regression_table(m_sar, m_sem; model_names = ["SAR", "SEM"]))

ρ̂ = 0.379 in SAR and λ̂ = 0.600 in SEM, both decisively non-zero. Look at what
the choice does to the density coefficient: **3.57 under SAR against 6.84 under
SEM**, nearly double. That is not a rounding difference, and it is the reason
the specification question is not academic. Under SAR part of density's
association with purchasing power is routed through the spatial multiplier and
attributed to neighbours; under SEM none of it is, so the direct coefficient
has to carry the whole association.

The ageing coefficient is −0.024 per point of the ratio. Per *hundred* points —
roughly the spread between a young municipality and a middling one — that is
−2.4 index points.

In [ ]:
println(model_comparison_table(m_sar, m_sdm, m_sem; names = ["SAR", "SDM", "SEM"]))
println(vuong_test(m_sar, m_sem))

**The diagnostics were right.** SEM beats SAR by 34 AIC points (2082.0 against
2115.9) and the Vuong test prefers SEM with a statistic of −3.76, p = 0.00017.
The robust LM pair predicted that before anything was estimated, which is the
best advertisement the battery gets in this package — and worth contrasting
with notebook `04`, where on 49 Columbus neighbourhoods the same battery lands
just short of the 5% critical value and settles nothing.

Then SDM wins outright on AIC (2071.8), beating SEM by 10 points with two extra
parameters. The honest reading is that neither pure specification is adequate:
once the neighbours' *regressors* are allowed in, they earn their place. That
is a familiar outcome and it is a warning against reporting a single preferred
model as though the data had chosen it cleanly. BIC, which penalises harder,
puts SDM at 2093.5 against SEM's 2096.5 — still SDM, but by three points
rather than ten.

In [ ]:
println(spatial_effects(m_sar))

In a model with a spatial lag the coefficient is not the effect. A change in
one municipality propagates to its neighbours, back again, and outward, so the
total is the coefficient scaled by the spatial multiplier.

For log density the direct effect is 3.69 and the **indirect effect is 2.06**,
giving a total of 5.75 — more than half again the 3.57 in the coefficient
table. Reading that table as though it held marginal effects would understate
the association by 38%. The same applies to ageing: −0.025 direct, −0.014
indirect, −0.039 total.

Note that the SEM has no such decomposition, by construction: dependence in the
disturbance affects efficiency and inference, not the propagation of a change
in `X`. If you prefer SEM on fit, you lose the spillover story — which is a
substantive consequence of a specification choice, not a technicality.

## 5. Does one model fit the whole country?

Everything so far assumed a single set of coefficients from Viana do Castelo to
Faro. Portugal is the standard example of a country where that is doubtful: the
*litoral* is dense, younger and richer, the interior sparse, old and poorer.

The split has to be declared **before** looking at the fit, or the test that
follows means nothing. Coastal here means the nine districts with an Atlantic
or estuarine coastline. Disagree with the list if you like — that is the point
of writing it out.

In [ ]:
const COASTAL = Set(["Viana do Castelo", "Braga", "Porto", "Aveiro", "Coimbra",
                     "Leiria", "Lisboa", "Setúbal", "Faro"])
labels = [d in COASTAL ? 1 : 2 for d in distr]

@printf("coastal %d municipalities, interior %d\n",
        count(==(1), labels), count(==(2), labels))
@printf("mean purchasing power   coastal %6.2f   interior %6.2f\n",
        mean(y[labels .== 1]), mean(y[labels .== 2]))
@printf("mean ageing ratio       coastal %6.1f   interior %6.1f\n",
        mean(X[labels .== 1, 2]), mean(X[labels .== 2, 2]))

The split lands on **139 and 139**, which is a coincidence and a convenient
one. Coastal municipalities average 88.2 against the interior's 77.6, and the
ageing ratio averages 211 against 306 — the interior has half again as many
pensioners per child.

Those gaps are descriptive. The question is whether the *relationship* differs,
which is what a regime model asks.

In [ ]:
m_reg = fit_spatial_regime(W, X, y, ExogenousRegimes(labels);
                           varnames = vn, method = ExactLU())
println(m_reg)
println(regime_chow_test(m_sar, m_reg))

**It rejects.** LR = 14.39 on 3 degrees of freedom, p = 0.0024. One set of
coefficients does not fit both halves of the country.

And the way it fails is more interesting than the fact that it does:

| | coastal | interior |
|---|---|---|
| log density | **6.01** (z = 6.8) | 2.07 (p = 0.048) |
| ageing ratio | −0.0004 (p = **0.98**) | **−0.042** (p < 0.0001) |

On the coast, purchasing power tracks density and the ageing ratio is
**indistinguishable from zero** — the coefficient is −0.0004, which is not a
weak effect but no effect at all. Inland, ageing is the dominant term and
density barely clears 5%. The two halves of Portugal are not the same economy
with different intercepts; they have different structures, and a pooled model
reports an average of two mechanisms that do not apply in the same place.

ρ̂ falls from 0.379 to 0.338 once the regimes absorb some of the clustering,
which is the expected direction: part of what the spatial lag was fitting was
heterogeneity in the coefficients, not contagion between neighbours.

Contrast notebook `05`, which runs the same test on Columbus's north/south
dummy and **fails to reject** — the likelihood barely moves and three extra
parameters are not earned. Same test, same package, opposite conclusion. The
difference is 278 units against 49 and a split that corresponds to something.

In [ ]:
sk = skater_regimes(W, hcat(y, X), 3; min_size = 10, rng = MersenneTwister(42))
@printf("SKATER regime sizes: %s\n", [count(==(k), sk.labels) for k in 1:3])

m_sk = fit_spatial_regime(W, X, y, sk; varnames = vn, method = ExactLU())
println(regime_chow_test(m_sar, m_sk))

Now do it the tempting way, and treat the result with the suspicion it deserves.

SKATER finds contiguous regions that are internally homogeneous, and it is
given `y` to do it. The Chow test then returns LR = 35.9, p = 3e-6 — far
stronger than the coastal/interior split. **That number is not evidence of
anything.** The regions were chosen to maximise exactly the heterogeneity the
test then measures, so the null was never live. The sizes alone should prompt
caution: 228, 40 and 10, with the smallest only just above the `min_size = 10`
floor.

SKATER is a tool for *finding* a hypothesis. The coastal/interior split is a
tool for testing one. The p-value from the first is a description; only the
second is inference, and only because the districts were written down first.

## 6. Two choices that look consequential

### Queen or rook

Whether contiguity requires a shared edge or merely a shared point is presented
in every textbook as a decision to make carefully. On these boundaries it is
two links out of 739 — Mêda with Sernancelhe, and Lisboa with Vila Franca de
Xira, each pair meeting at a single point.

In [ ]:
m_sar_rook = fit_spatial(SAR(), Wr, X, y; varnames = vn, method = ExactLU())
@printf("rho   queen %.6f    rook %.6f    difference %.1e\n",
        m_sar.rho, m_sar_rook.rho, abs(m_sar.rho - m_sar_rook.rho))
@printf("Moran's I  queen %.6f    rook %.6f\n",
        moran_global(y, W).I_stat, moran_global(y, Wr).I_stat)

ρ̂ moves from 0.3786 to 0.3880, a difference of 0.009 against a standard error
of 0.068 — about an eighth of one. Moran's I agrees to three decimal
places: 0.445181 against 0.445115.

The lesson is not that the choice never matters. It is that on a **clean planar
partition of real administrative boundaries** the two criteria are nearly the
same graph, so the agonising is better spent elsewhere. On a lattice, or on
boundaries with slivers and gaps, the difference can be large.

### How wide is a Conley neighbourhood

The weights matrix is one way to model dependence; a nonparametric robust
covariance is another, and it needs a bandwidth rather than a graph. The
centroids are real coordinates in degrees, so a bandwidth of 1.0 is about
86 km east–west and 111 km north–south — and the whole country spans only
3.0° of longitude by 5.0° of latitude, so a wide bandwidth includes a large
share of the sample.

A single bandwidth is a choice nobody can defend. Sweep it.

In [ ]:
cl     = readlines(joinpath(DATA, "portugal_municipalities_centroids.csv"))
crows  = [split(strip(l), ',') for l in cl[2:end] if !isempty(strip(l))]
@assert [r[1] for r in crows] == dico   "centroid file is out of order"
coords = hcat([parse(Float64, r[3]) for r in crows],
              [parse(Float64, r[4]) for r in crows])

se_base = sqrt.(diag(vcov(m_sar)))
@printf("%-8s %12s %12s %10s\n", "bw(deg)", "se(log_dens)", "se(ageing)", "se(rho)")
@printf("%-8s %12.4f %12.4f %10.4f   <- model-based\n",
        "-", se_base[2], se_base[3], se_base[4])
for bw in (0.25, 0.5, 1.0, 1.5, 2.0, 3.0)
    se = sqrt.(diag(robust_vcov(m_sar, ConleySpatialHAC(coords, bw))))
    @printf("%-8.2f %12.4f %12.4f %10.4f\n", bw, se[2], se[3], se[4])
end

Two different stories in one table, which is why the sweep is worth running.

**The slopes are fine.** `se(log_density)` jumps from 0.63 model-based to
0.84–0.94 and drifts within that band; `se(ageing)` from 0.0079 to 0.0089–0.0106.
Both are a third to a half wider than the model-based figures — so those were
optimistic — but the widening lands in a band rather than running away, and the
ageing coefficient stays significant at 5% across every bandwidth, its z moving
from −3.11 to a worst case of −2.30. That conclusion is robust in the ordinary
sense of the word.

**ρ̂'s standard error is not settled at all.** It runs 0.036, 0.044, 0.064,
0.080, 0.094, 0.114 — monotone, and 3× the model-based 0.068 at the widest
bandwidth with no sign of a plateau. At the narrow end it is *below* the
model-based value, which is a warning rather than a result: too few pairs fall
inside the kernel for the middle of the sandwich to be estimated.

So report the slopes with Conley errors and be glad of it, and do not claim
Conley has pinned down the precision of ρ̂. A estimator whose robust standard
error triples across defensible bandwidths has not been made robust by
choosing one.

## What this notebook is for

- **Check the join, do not trust it.** `read_gal` returns its ids; comparing
  them against the attribute file's key costs one line and catches the error
  class that produces plausible numbers from misaligned rows.
- **Moran's I on a variable is not a reason to fit a spatial model.** Moran's I
  on the *residuals* is. Here it was 0.445 raw and still 0.336 after OLS
  absorbed density and age.
- **Read the robust LM pair, not the classical one.** It predicted SEM over SAR
  before estimation, and the likelihood and the Vuong test agreed.
- **In a lag model the coefficient is not the effect.** Density's total effect
  is 5.75 against a coefficient of 3.57.
- **A regime split declared in advance is inference; one found by SKATER is a
  description.** The second gave the far smaller p-value and means far less.
- **Sweep the nuisance parameter.** The slopes' robust errors converge; ρ̂'s
  triples. One table shows both.

## Exercises

1. The coastal district list is a judgement. Replace it with a rule on centroid
   longitude — say, west of −8.3° — and see whether the Chow test still
   rejects and whether the coefficient pattern survives. Which split would you
   defend to a referee, and on what grounds other than the result?
2. SDM won on AIC. Fit it, read `spatial_effects`, and compare the indirect
   effects with SAR's. Does allowing the neighbours' regressors in change the
   story about density, or only the fit?
3. Put the islands back, build `knn_contiguity`-style weights from the
   centroids so that every one of the 308 has neighbours, and re-estimate. Does
   ρ̂ move? Should a municipality in Madeira be a neighbour of anything in the
   Algarve?
4. `pp_share_national` and `pp_dynamism` are in the CSV and come from the same
   INE study as the outcome. Put one on the right-hand side and watch the fit
   improve. Explain, in one paragraph, why that improvement is worthless.
5. The ageing ratio has no measurable association with purchasing power on the
   coast and a strong one inland. Propose two mechanisms that would produce
   that, and say what additional data would separate them.